# G2 — Live demo: one head, five encoders, real images

This is the G1 result made visible. It loads the same cached spaces,
builds the same 512-d hub, trains the caption head on **DINOv2-small
only** — and then shows, for real COCO images you can look at, the same
head retrieving captions through encoders it has never seen.

Nothing here is simulated: the vectors are the cached ones the measured
numbers came from, and the captions and images are re-derived from the
COCO annotations by the same deterministic recipe E1 used (captions
grouped per image, ids sorted, prefix taken) — so row *i* of every cache
is image `ids[i]`, and what you see on screen is what the numbers were
measured on.

In [ ]:
import os
from pathlib import Path
STORAGE   = "drive"                     # "drive" | "local" | "env"
DRIVE_DIR = "/content/drive/MyDrive/convergence_experiment"
LOCAL_DIR = "./convergence_data"
try:
    import google.colab                 # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if STORAGE == "env":
    assert os.environ.get("DATA_DIR"), "STORAGE='env' but DATA_DIR unset"
elif STORAGE == "drive" and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["DATA_DIR"] = DRIVE_DIR
else:
    if STORAGE == "drive":
        print("not on Colab - using LOCAL_DIR")
    os.environ["DATA_DIR"] = str(Path(LOCAL_DIR).resolve())
DATA_DIR = Path(os.environ["DATA_DIR"])
print("DATA_DIR:", DATA_DIR)

In [ ]:
import numpy as np, json, zipfile

# ---- vectors: same caches G1 used ----
SPACES = {}
for size in ("small", "base", "large"):
    f = DATA_DIR / f"e1_img_ckpt_dinov2-{size}_cls+patch.npz"
    if f.exists():
        SPACES[f"img_{size}"] = np.load(str(f))["img"].astype(np.float64)
d = np.load(str(DATA_DIR / "crossmodal_pairs.npz"))
SPACES["txt_bge"] = d["txt"].astype(np.float64)
# alignment proof, as in G1: the pairs file's image block must equal a cache
_ok = False
for ref in [k for k in SPACES if k.startswith("img_")]:
    if d["img"].shape[1] == SPACES[ref].shape[1]:
        n = min(len(d["img"]), len(SPACES[ref]))
        if np.allclose(d["img"][:n], SPACES[ref][:n], atol=1e-4):
            _ok = True
assert _ok, "pairs file aligns with no image cache - stop"
N = min(len(v) for v in SPACES.values())
SPACES = {k: v[:N] for k, v in SPACES.items()}
print({k: v.shape for k, v in SPACES.items()})

# ---- captions + images: re-derive ids by E1's exact recipe ----
zf = DATA_DIR / "annotations_trainval2017.zip"
assert zf.exists(), "annotations zip not on Drive - run E1/B1 once first"
with zipfile.ZipFile(str(zf)) as z:
    ann = json.load(z.open("annotations/captions_train2017.json"))
url  = {im["id"]: im["coco_url"] for im in ann["images"]}
caps = {}
for a in ann["annotations"]:
    caps.setdefault(a["image_id"], []).append(a["caption"].strip())
ids = sorted(set(caps) & set(url))[:N]
assert len(ids) == N, "id derivation shorter than the caches - stop"
print(f"{N} rows; row i of every cache = image ids[i]")

rng = np.random.default_rng(0)                # SAME split as G1
perm = rng.permutation(N)
te, tr = perm[:1000], perm[1000:]

In [ ]:
# ---- the hub and the single head, exactly as measured ----
HUB_DIM, ALPHA = 512, 1e-2
def l2n(V): return V / (np.linalg.norm(V, axis=-1, keepdims=True) + 1e-12)
def ridge(X, Y, a=ALPHA):
    return np.linalg.solve(X.T @ X + a * np.eye(X.shape[1]), X.T @ Y)

_ref = np.hstack([(SPACES[k][tr] - SPACES[k][tr].mean(0)) /
                  (SPACES[k][tr].std(0).mean() + 1e-12) for k in SPACES])
_mu = _ref.mean(0)
_U, _sv, _VT = np.linalg.svd(_ref - _mu, full_matrices=False)
BASIS = _VT[:HUB_DIM].T / (_sv[:HUB_DIM] / np.sqrt(len(_ref)))
HUB_TR = (_ref - _mu) @ BASIS
TO_HUB = {k: ridge(v[tr], HUB_TR) for k, v in SPACES.items()}

TRAIN_ON = "img_small"
HEAD = ridge(SPACES[TRAIN_ON][tr] @ TO_HUB[TRAIN_ON], SPACES["txt_bge"][tr])
GAL = l2n(SPACES["txt_bge"][te])              # 1000-caption gallery

def recall_at_1(enc):
    P = l2n((SPACES[enc][te] @ TO_HUB[enc]) @ HEAD)
    return float(((P @ GAL.T).argmax(1) == np.arange(len(te))).mean())

donors = [k for k in SPACES if k.startswith("img_")]
print(f"head trained ONLY on {TRAIN_ON}; sanity vs the measured table:")
for e in donors:
    print(f"   {e:10s} R@1 {recall_at_1(e):.3f}"
          + ("   <- trained here" if e == TRAIN_ON else "   <- never seen"))

## The demo

Each example shows a real held-out COCO image, then the top-3 captions
retrieved by the **same head** through each encoder — including the two
it never saw. A check mark means the true caption of that image ranked
first among all 1,000 candidates.

In [ ]:
import urllib.request
from io import BytesIO
try:
    from PIL import Image as PILImage
    from IPython.display import display
    _CAN_SHOW = True
except ImportError:
    _CAN_SHOW = False

def show_examples(n_examples=3, seed=None):
    r = np.random.default_rng(seed)
    picks = r.choice(len(te), n_examples, replace=False)
    for q in picks:
        img_id = ids[te[q]]
        print("=" * 72)
        print(f"image {img_id}  -  true caption:")
        print(f'   "{caps[img_id][0]}"')
        if _CAN_SHOW:
            try:
                raw = urllib.request.urlopen(url[img_id], timeout=10).read()
                im = PILImage.open(BytesIO(raw)); im.thumbnail((360, 360))
                display(im)
            except Exception as e:
                print(f"   (image fetch failed: {type(e).__name__} - "
                      f"{url[img_id]})")
        for enc in donors:
            P = l2n((SPACES[enc][te[q:q+1]] @ TO_HUB[enc]) @ HEAD)
            order = np.argsort(-(P @ GAL.T).ravel())[:3]
            hit = "CORRECT @1" if order[0] == q else "          "
            tag = "trained here" if enc == TRAIN_ON else "NEVER SEEN"
            print(f"\n   {enc}  ({tag})   {hit}")
            for rk, g in enumerate(order, 1):
                mark = " <-- true" if g == q else ""
                print(f"      {rk}. {caps[ids[te[g]]][0][:70]}{mark}")
    print("=" * 72)
    print("one head, trained on img_small only - the other encoders were")
    print("never seen by it, in any form, at any point.")

show_examples(3, seed=7)

## The control, live

Replace the fitted hub map with a random one of the same shape. If the
demo above worked because the head is somehow doing the job unaided,
this will work too. It will not.

In [ ]:
def show_control(seed=7):
    r = np.random.default_rng(seed)
    q = int(r.integers(len(te)))
    enc = donors[-1]
    R = np.random.default_rng(99).standard_normal(TO_HUB[enc].shape) / \
        np.sqrt(SPACES[enc].shape[1])
    img_id = ids[te[q]]
    print(f'true caption: "{caps[img_id][0]}"\n')
    P = l2n((SPACES[enc][te[q:q+1]] @ R) @ HEAD)
    order = np.argsort(-(P @ GAL.T).ravel())[:3]
    print(f"{enc} through a RANDOM map - top 3 of 1000:")
    for rk, g in enumerate(order, 1):
        print(f"   {rk}. {caps[ids[te[g]]][0][:70]}")
    print("\nunrelated captions: the transfer above is carried by the")
    print("fitted hub geometry, not by the head alone.")

show_control()